# Huấn luyện CDNMF trên dữ liệu nghệ sĩ (Kaggle)

Notebook này dùng `artist_csb.csv` làm dữ liệu thuộc tính và `edges.csv` làm đồ thị nghệ sĩ. Notebook tự tìm hai CSV trong `/kaggle/input` hoặc thư mục làm việc, sau đó huấn luyện CDNMF qua mã nguồn repository.

**Trên Kaggle:** tạo notebook, chọn **Add Input** để đính kèm dataset có cả hai CSV; bật **Settings → Internet** để tải mã CDNMF từ GitHub. Bật GPU nếu muốn. Chạy lần lượt từ trên xuống. Có thể giảm `RUNS`, `EPOCHS`, `PRETRAIN_MAX_ITER` để kiểm tra nhanh trước.

Notebook tạo random-walk pairs từ đồ thị và dùng chúng trong hạng smoothness của loss. Nhãn đánh giá lấy từ `department_title` nếu cột này có trong CSV, nếu không dùng `main_department` (cột hiện có trong file). Nhãn này chỉ dùng để tính NMI/ARI, không được đưa vào đặc trưng đầu vào. Modularity được tính trên đồ thị có trọng số.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Kaggle thường đã cài sẵn các thư viện này và PyTorch có CUDA; không cài đè PyTorch.
required = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn", "networkx": "networkx", "matplotlib": "matplotlib", "tqdm": "tqdm"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

import torch

REPOSITORY_URL = "https://github.com/HongHa-1606/CDNMF.git"
START_DIR = Path.cwd()
KAGGLE_WORKING = Path("/kaggle/working")
REPO_DIR = KAGGLE_WORKING / "CDNMF" if KAGGLE_WORKING.exists() else START_DIR
if not (REPO_DIR / "Model" / "my_model.py").is_file():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        REPO_DIR = START_DIR / "CDNMF"
    if not (REPO_DIR / "Model" / "my_model.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print(f"Mã CDNMF: {REPO_DIR}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if missing:
    print("Đã cài thư viện còn thiếu:", ", ".join(missing))

In [ ]:
import json
import random
import tempfile
import time
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from Model.my_model import Model
from PreTrainer.pretrainer import PreTrainer

DATA_FILES_ROOTS = [Path("/kaggle/input"), START_DIR, Path.cwd()]
def find_csv(filename):
    for root in DATA_FILES_ROOTS:
        if root.exists():
            matches = sorted(path for path in root.rglob(filename) if path.is_file())
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f"Không tìm thấy {filename}. Hãy đính kèm Kaggle Input chứa file này hoặc đặt file trong thư mục làm việc."
    )

artists_path = find_csv("artist_csb.csv")
edges_path = find_csv("edges.csv")
artists = pd.read_csv(artists_path, dtype={"artist_id": str})
edges = pd.read_csv(edges_path, dtype={"source": str, "target": str})
required_artist_columns = {"artist_id"}
required_edge_columns = {"source", "target", "weight"}
if not required_artist_columns.issubset(artists.columns):
    raise ValueError(f"artist_csb.csv thiếu cột: {required_artist_columns - set(artists.columns)}")
if not required_edge_columns.issubset(edges.columns):
    raise ValueError(f"edges.csv thiếu cột: {required_edge_columns - set(edges.columns)}")
if artists["artist_id"].duplicated().any():
    raise ValueError("artist_id phải duy nhất trong artist_csb.csv.")
if edges["weight"].isna().any() or not np.isfinite(edges["weight"].to_numpy(dtype=float)).all():
    raise ValueError("Cột weight có giá trị thiếu hoặc không hữu hạn.")
if (edges["weight"] < 0).any():
    raise ValueError("CDNMF yêu cầu trọng số cạnh không âm.")

artist_ids = artists["artist_id"].astype(str).tolist()
node_index = {artist_id: i for i, artist_id in enumerate(artist_ids)}
edges = edges.loc[edges["weight"] > 0].copy()
known = edges["source"].isin(node_index) & edges["target"].isin(node_index)
unknown_edge_count = int((~known).sum())
edges = edges.loc[known & (edges["source"] != edges["target"])].copy()
rows = edges["source"].map(node_index).to_numpy(dtype=int)
cols = edges["target"].map(node_index).to_numpy(dtype=int)
weights = edges["weight"].to_numpy(dtype=np.float32)
A_sparse = sp.coo_matrix((weights, (rows, cols)), shape=(len(artists), len(artists))).tocsr()
A_sparse.sum_duplicates()
A_sparse = A_sparse.maximum(A_sparse.T).tocsr()
A_sparse.setdiag(0)
A_sparse.eliminate_zeros()
A = A_sparse.toarray().astype(np.float32, copy=False)
print(f"Nghệ sĩ: {len(artists):,} | Cạnh vô hướng: {A_sparse.nnz // 2:,} | Cạnh bỏ do artist không có trong CSV: {unknown_edge_count:,}")
print(f"CSV nghệ sĩ: {artists_path}\nCSV cạnh: {edges_path}")

In [ ]:
def tag_words(value):
    if pd.isna(value) or not str(value).strip():
        return ""
    try:
        parsed = json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return str(value)
    if isinstance(parsed, dict):
        return " ".join(str(key).replace("_", " ") for key in parsed)
    return str(parsed)

text_fields = [column for column in ["subject_tags", "style_tags", "technique_tags", "material_tags"] if column in artists.columns]
documents = []
for _, row in artists.iterrows():
    tokens = [tag_words(row[column]) for column in text_fields]
    for column in ["main_century", "main_place_of_origin"]:
        if column in artists.columns and pd.notna(row[column]):
            tokens.append(f"{column} {row[column]}")
    documents.append(" ".join(token for token in tokens if token))

# Giới hạn số chiều để mô hình dense phù hợp bộ nhớ Kaggle; nhãn phòng ban không nằm trong text.
vectorizer = TfidfVectorizer(max_features=256, min_df=1, token_pattern=r"(?u)\b\w+\b")
text_features = vectorizer.fit_transform(documents).astype(np.float32)
numeric_features = np.zeros((len(artists), 1), dtype=np.float32)
if "artwork_count" in artists.columns:
    artwork_count = pd.to_numeric(artists["artwork_count"], errors="coerce").fillna(0).to_numpy(dtype=np.float32)
    numeric_features[:, 0] = np.log1p(np.maximum(artwork_count, 0))
    maximum = numeric_features.max()
    if maximum > 0:
        numeric_features /= maximum
X_sparse = sp.hstack([text_features, sp.csr_matrix(numeric_features)], format="csr")
X = X_sparse.toarray().astype(np.float32, copy=False)
if X.shape[1] < 2 or not np.isfinite(X).all():
    raise ValueError(f"Ma trận đặc trưng không hợp lệ: shape={X.shape}")

LABEL_COLUMN = "department_title" if "department_title" in artists.columns else "main_department"
if LABEL_COLUMN not in artists.columns:
    raise ValueError("Cần cột department_title hoặc main_department để đánh giá NMI/ARI.")
label_series = artists[LABEL_COLUMN]
labeled_mask = label_series.notna().to_numpy()
label_codes = pd.Categorical(label_series[labeled_mask]).codes
if len(np.unique(label_codes)) < 2:
    raise ValueError(f"Cột nhãn {LABEL_COLUMN} cần có ít nhất hai phòng ban khác nhau.")
print(f"Ma trận thuộc tính: {X.shape} | Nhãn đánh giá: {LABEL_COLUMN} ({len(np.unique(label_codes))} lớp)")

# CDNMF dùng tensor dense; giữ dữ liệu trên GPU khi có GPU và đủ dung lượng.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
graph = type("ArtistGraph", (), {})()
graph.A = torch.as_tensor(A, dtype=torch.float32, device=device)
graph.X = torch.as_tensor(X, dtype=torch.float32, device=device)
graph.L = torch.as_tensor(np.where(labeled_mask, pd.Categorical(label_series).codes, -1), dtype=torch.float32, device=device)
graph.num_nodes = len(artists)
graph.num_feas = X.shape[1]
WALK_NUM_WALKS = 5
WALK_LENGTH = 40
WALK_WINDOW = 5
walk_rng = np.random.default_rng(42)
neighbors = [A_sparse.indices[A_sparse.indptr[i]:A_sparse.indptr[i + 1]].tolist() for i in range(len(artists))]
walk_pairs = set()
for start_node in range(len(artists)):
    for _ in range(WALK_NUM_WALKS):
        walk = [start_node]
        while len(walk) < WALK_LENGTH and neighbors[walk[-1]]:
            walk.append(int(walk_rng.choice(neighbors[walk[-1]])))
        for position, node in enumerate(walk):
            for context in walk[max(0, position - WALK_WINDOW):min(len(walk), position + WALK_WINDOW + 1)]:
                if context != node:
                    walk_pairs.add((min(node, context), max(node, context)))
walk_rows, walk_cols = zip(*sorted(walk_pairs)) if walk_pairs else ([], [])
W_sparse = sp.coo_matrix(
    (np.ones(len(walk_rows), dtype=np.float32), (walk_rows, walk_cols)),
    shape=(len(artists), len(artists)),
).tocsr()
W_sparse = W_sparse.maximum(W_sparse.T).tocsr()
graph.W = torch.as_tensor(W_sparse.toarray(), dtype=torch.float32, device=device)
if not walk_pairs:
    raise ValueError("Không tạo được walk pairs; kiểm tra đồ thị có cạnh hay không.")
print(f"Walk pairs: {len(walk_pairs):,} | walks/node: {WALK_NUM_WALKS} | length: {WALK_LENGTH} | window: {WALK_WINDOW}")
print("Thiết bị huấn luyện:", device)

## Cấu hình và khởi tạo CDNMF

Mặc định mô hình học `K` cộng đồng. Khởi tạo NMF theo hai tầng được sinh trực tiếp từ ma trận đồ thị và ma trận thuộc tính, nên không phụ thuộc các file pretrain của Cora/Citeseer/PubMed. Mỗi lần chạy làm nhiễu nhân nhỏ các hệ số khởi tạo theo seed riêng để phép đo độ ổn định phản ánh nhiều lần tối ưu độc lập.

In [ ]:
RUNS = 10
EPOCHS = 200
K = 8
PRETRAIN_MAX_ITER = 100
SEEDS = [42 + i for i in range(RUNS)]
LEARNING_RATE = 0.001

if RUNS < 2 or EPOCHS < 1 or K < 2 or len(SEEDS) != RUNS:
    raise ValueError("Cần RUNS >= 2, EPOCHS >= 1, K >= 2 và một seed cho mỗi run.")
net_shape = [128, K]
att_shape = [64, K]
if K > min(net_shape[0], att_shape[0], len(artists), X.shape[1]):
    raise ValueError("K phải nhỏ hơn hoặc bằng số chiều khả dụng của dữ liệu.")

with tempfile.TemporaryDirectory(prefix="artist_cdnmf_") as temp_dir:
    pretrain_path = Path(temp_dir) / "pretrain_params.pkl"
    pretrain_config = {
        "net_shape": net_shape,
        "att_shape": att_shape,
        "net_input_dim": len(artists),
        "att_input_dim": X.shape[1],
        "seed": SEEDS[0],
        "pre_iterations": PRETRAIN_MAX_ITER,
        "pretrain_params_path": str(pretrain_path),
    }
    pretrainer = PreTrainer(pretrain_config)
    pretrainer.pre_training(A, "net")
    pretrainer.pre_training(X.T, "att")
    base_U, base_V = deepcopy(pretrainer.U_init), deepcopy(pretrainer.V_init)
    del pretrainer
    print(f"Đã tạo khởi tạo NMF; số chiều network/attribute = {net_shape[0]}/{att_shape[0]}.")

    network = nx.Graph()
    network.add_nodes_from(range(len(artists)))
    coo = A_sparse.tocoo()
    network.add_weighted_edges_from(
        (int(i), int(j), float(weight))
        for i, j, weight in zip(coo.row, coo.col, coo.data)
        if i < j
    )
    if network.number_of_edges() == 0:
        raise ValueError("Đồ thị không có cạnh sau khi lọc; không thể tính modularity.")

    run_rows = []
    predictions = []
    run_seconds = []
    for run_index, seed in enumerate(SEEDS, start=1):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        rng = np.random.default_rng(seed)
        run_U, run_V = deepcopy(base_U), deepcopy(base_V)
        for params in (run_U, run_V):
            for name, value in params.items():
                if name.endswith("1"):
                    params[name] = np.maximum(value * np.exp(rng.normal(0, 0.05, value.shape)).astype(np.float32), 1e-8)
        with open(pretrain_path, "wb") as handle:
            import pickle
            pickle.dump([run_U, run_V], handle, protocol=pickle.HIGHEST_PROTOCOL)

        model_config = {
            "device": device,
            "net_shape": net_shape,
            "att_shape": att_shape,
            "net_input_dim": len(artists),
            "att_input_dim": X.shape[1],
            "is_init": True,
            "pretrain_params_path": str(pretrain_path),
            "tau": 1.0,
            "conc": 1.0,
            "negc": 1.0,
            "rec": 1.0,
            "r": 1.0,
            "walk_weight": 1.0,
            "learning_rate": LEARNING_RATE,
            "weight_decay": 0.0,
            "epoch": EPOCHS,
            "run": RUNS,
            "model_path": str(Path(temp_dir) / f"model_{seed}.pkl"),
        }
        model = Model(model_config).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        start_time = time.perf_counter()
        model.train()
        for epoch in range(1, EPOCHS + 1):
            optimizer.zero_grad(set_to_none=True)
            model()
            loss, *_ = model.loss(graph)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss không hữu hạn ở seed={seed}, epoch={epoch}.")
            loss.backward()
            optimizer.step()
        elapsed = time.perf_counter() - start_time
        run_seconds.append(elapsed)

        model.eval()
        with torch.no_grad():
            memberships = model().detach().cpu().numpy()
        assignment = np.argmax(memberships, axis=0)
        communities = [set(np.flatnonzero(assignment == community)) for community in np.unique(assignment)]
        sizes = sorted((len(community) for community in communities), reverse=True)
        modularity = nx.algorithms.community.modularity(network, communities, weight="weight")
        nmi = normalized_mutual_info_score(label_codes, assignment[labeled_mask])
        ari = adjusted_rand_score(label_codes, assignment[labeled_mask])
        run_rows.append({
            "seed": seed,
            "loss": float(loss.detach().cpu()),
            "modularity": modularity,
            "NMI_department": nmi,
            "ARI_department": ari,
            "communities": len(sizes),
            "singletons": sum(size == 1 for size in sizes),
            "size_2_communities": sum(size == 2 for size in sizes),
            "communities_size_le_2_pct": 100 * sum(size <= 2 for size in sizes) / len(sizes),
            "largest_community_pct": 100 * sizes[0] / len(artists),
            "runtime_seconds": elapsed,
        })
        predictions.append(assignment)
        print(f"Run {run_index}/{RUNS} | seed={seed} | loss={loss.item():.4f} | modularity={modularity:.4f} | NMI={nmi:.4f} | ARI={ari:.4f} | {elapsed:.1f}s")
        del model, optimizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results = pd.DataFrame(run_rows)
    pairwise_nmi = np.eye(RUNS, dtype=float)
    for i in range(RUNS):
        for j in range(i + 1, RUNS):
            score = normalized_mutual_info_score(predictions[i], predictions[j])
            pairwise_nmi[i, j] = pairwise_nmi[j, i] = score

    summary = results.drop(columns="seed").agg(["mean", "std"]).T
    summary.columns = ["mean", "std"]
    summary.loc["stability_pairwise_NMI", "mean"] = pairwise_nmi[np.triu_indices(RUNS, k=1)].mean()
    summary.loc["stability_pairwise_NMI", "std"] = pairwise_nmi[np.triu_indices(RUNS, k=1)].std()
    print("\nKết quả từng lần chạy:")
    display(results.round(4))
    print("\nTrung bình ± độ lệch chuẩn:")
    display(summary.round(4))
    print(f"NMI giữa các lần chạy (trung bình ± std): {summary.loc['stability_pairwise_NMI', 'mean']:.4f} ± {summary.loc['stability_pairwise_NMI', 'std']:.4f}")

    size_counts = pd.Series(predictions[0]).value_counts().sort_values(ascending=False)
    plt.figure(figsize=(10, 4))
    plt.bar(np.arange(1, len(size_counts) + 1), size_counts.to_numpy())
    plt.xlabel("Cộng đồng (xếp theo kích thước giảm dần)")
    plt.ylabel("Số nghệ sĩ")
    plt.title(f"Phân bố kích thước cộng đồng — seed {SEEDS[0]}")
    plt.tight_layout()
    plt.show()

    output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else START_DIR
    results.to_csv(output_dir / "artist_cdnmf_metrics.csv", index=False)
    pd.DataFrame({"artist_id": artist_ids, "community": predictions[0]}).to_csv(output_dir / "artist_cdnmf_assignments.csv", index=False)
    pd.DataFrame(pairwise_nmi, index=SEEDS, columns=SEEDS).to_csv(output_dir / "artist_cdnmf_stability_nmi.csv")
    print(f"Đã lưu metrics, phân cụm seed đầu tiên và ma trận NMI vào: {output_dir}")